# MJO RMM skill of the hindcasts

## Purpose

Verify the MJO in the S0 (Jan-Feb 2012) and S1 (Jun-Jul 2012) hindcast ensembles: every member and the observations are projected onto the same observed RMM EOFs, so model and observed RMM are one index.

## Inputs

- Observed RMM EOFs (`RMM_EOF_FILE`), built by `hind_atm_mjo/03_compute_and_plot_rmm_skill.ipynb` (NOAA OLR, ERA5 U850/U200, 1986-2017, aligned with the BoM index); copy it to this machine or rebuild it here.
- Hindcast members (`util/bias_metrics.py` registry, `<run>/archive/post/atm/180x360_aave/ts`): daily-mean FLUT and daily U850/U200 estimated as the mean of the 00Z states bracketing each day (`util/model_daily.py`).
- Observations under `OBSERVATION_ROOT` (the reference-data archive or its per-machine copy): NOAA OLR daily, ERA5 6-hourly U850/U200 averaged to days, from `HISTORY_DAYS` before each start date; the BoM index (`MJO_INDEX`) as a cross-check.

## Method

Same preprocessing as the EOF training (EOF-file climatology and normalisation, previous-120-day mean removed, sqrt(cos lat) weighting). The previous-120-day mean of a hindcast uses the observed anomalies before its start date (Gottschalck et al. 2010), so every lead day is usable. Scores (Rashid et al. 2011), with one start date per season:

- per lead day: ensemble-mean RMM error, ensemble spread, amplitude and phase errors of the ensemble mean;
- per lead window: bivariate COR and RMSE (means not removed) pooled over the window's days, for the ensemble mean and averaged over members.

A climatological forecast has error √2; spread below the ensemble-mean error means an under-dispersive ensemble.

## Outputs

`data/fcst_atm_mjo/rmm/rmm_<EXPERIMENT>.nc` (member x time), `rmm_OBS_<season>.nc`, `rmm_scores_<season>.nc`, `rmm_window_skill.csv`; figures in `figure/fcst_atm_mjo/rmm/`.

## 1. Libraries

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

## 2. User setup

Paths and run options. Edit this cell for normal changes; it holds plain values only, so `papermill -p` overrides work.

In [ ]:
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/compyfs/zhan391/acme_init/Observations")  # observations (shared archive)
WORKFLOW_NAME = "fcst_atm_mjo"

CASES = {
    "S0": ("CTRL10-S0", "CAPT10-S0", "DART40-S0"),  # start 2012-01-01
    "S1": ("CTRL10-S1", "CAPT10-S1", "DART40-S1"),  # start 2012-06-01
}
INIT_DATES = {"S0": "2012-01-01", "S1": "2012-06-01"}
RMM_EOF_FILE = DIAGNOSTIC_OUTPUT_ROOT / "data/hind_atm_mjo/rmm/rmm_obs_eof_1986-2017_climo31_rm120.nc"
BOM_RMM_FILE = OBSERVATION_ROOT / "MJO_INDEX/rmm.74toRealtime.txt"
HISTORY_DAYS = 150     # observed days before each start (>= 120)
MAX_LEAD = 59          # days shown
MAX_MEMBERS = None     # e.g. 5 for a quick test
FORCE_COMPUTE = False
SHOW = True
FIGURE_DPI = 300

## 3. Paths and modules

In [ ]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.dask_helpers import configure_dask

configure_dask(use_distributed=False)
data_dir, figure_dir = workflow_output_dirs(WORKFLOW_NAME, 'rmm', output_root=DIAGNOSTIC_OUTPUT_ROOT)
ROOTS = dict(observation_root=OBSERVATION_ROOT)
print(f"[DATA] {data_dir}\n[FIGURE] {figure_dir}")

def save(fig, name):
    path = figure_dir / name
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"[FIGURE] {path}")
    if not SHOW:
        plt.close(fig)

## 4. Driver: project and save

Observed RMM for each season, then every member of every experiment (cached per experiment).

In [ ]:
from util.bias_metrics import experiment_registry
from util.hind_observation_data import open_reference_daily
from util.mjo_rmm_projection import RmmProjector, open_forecast_member_fields, read_bom_rmm

if not Path(RMM_EOF_FILE).exists():
    raise FileNotFoundError(f"{RMM_EOF_FILE} not found: build it with hind_atm_mjo/03_compute_and_plot_rmm_skill.ipynb")
proj = RmmProjector(xr.load_dataset(RMM_EOF_FILE))
bom = read_bom_rmm(BOM_RMM_FILE)
products = {"FLUT": "NOAA_OLR", "U850": "ERA5_DAILY", "U200": "ERA5_DAILY"}
obs_rmm, fcst_rmm = {}, {}
for season, experiments in CASES.items():
    init = np.datetime64(INIT_DATES[season])
    start, end = str(init - np.timedelta64(HISTORY_DAYS, "D")), str(init + np.timedelta64(MAX_LEAD + 5, "D"))
    obs_fields = {v: open_reference_daily(p, v, start=start, end=end, lat_range=(-20.0, 20.0), **ROOTS)
                  for v, p in products.items()}
    history = {v: f.sel(time=slice(None, init - np.timedelta64(1, "D"))) for v, f in obs_fields.items()}
    obs_rmm[season] = proj.project(obs_fields, "OBS")
    obs_rmm[season].to_netcdf(data_dir / f"rmm_OBS_{season}.nc")
    registry = experiment_registry(INPUT_DATA_ROOT, season, "fc")
    for exp in experiments:
        out = data_dir / f"rmm_{exp}.nc"
        if out.exists() and not FORCE_COMPUTE:
            fcst_rmm[exp] = xr.load_dataset(out)
            print(f"[REUSE] {out}")
            continue
        years = range(int(INIT_DATES[season][:4]), int(end[:4]) + 1)
        members = open_forecast_member_fields(registry[exp], years, max_members=MAX_MEMBERS)
        fcst_rmm[exp] = proj.project_members(members, exp, history=history)
        fcst_rmm[exp].to_netcdf(out)
        print(f"[WRITE] {out}")

## 5. Scores

Per lead day and per lead window; the observed projection is compared with the BoM index as a check.

In [ ]:
from util.mjo_rmm_projection import bivariate_cor, rmm_ensemble_scores, rmm_window_skill

tables = []
for season, experiments in CASES.items():
    b, o = xr.align(bom[["RMM1", "RMM2"]].dropna("time"), obs_rmm[season], join="inner")
    print(f"[CHECK] {season}: observed RMM vs BoM, bivariate COR = "
          f"{bivariate_cor(o.RMM1.values, o.RMM2.values, b.RMM1.values, b.RMM2.values):.2f} over {b.sizes['time']} days")
    scores = xr.concat([rmm_ensemble_scores(fcst_rmm[e], obs_rmm[season], INIT_DATES[season]).expand_dims(experiment=[e])
                        for e in experiments], "experiment")
    scores.to_netcdf(data_dir / f"rmm_scores_{season}.nc")
    tables += [rmm_window_skill(fcst_rmm[e], obs_rmm[season], INIT_DATES[season]).assign(season=season, experiment=e)
               for e in experiments]
window_skill = pd.concat(tables)
window_skill.to_csv(data_dir / "rmm_window_skill.csv", index=False)
display(window_skill.set_index(["season", "experiment", "window"]).round(2))

## 6. Plot and show

In [ ]:
from configs.experiment_labels import experiment_label
from util.mjo_rmm_projection import plot_ensemble_rmm

colors = {"CTRL": "0.45", "CAPT": "tab:blue", "DART": "tab:red"}
for season, experiments in CASES.items():
    fig = plot_ensemble_rmm({experiment_label(e): fcst_rmm[e] for e in experiments}, obs_rmm[season],
                            INIT_DATES[season], max_lead=MAX_LEAD,
                            colors={experiment_label(e): colors[e[:4]] for e in experiments},
                            title=f"{season}: RMM hindcasts from {INIT_DATES[season]}")
    save(fig, f"rmm_ensemble_{season}.png")